# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


**Finding 1:** The dataset reveals that the 90-day window from the `fact_content_query_90d` table can overlap with the final months of data capture[cite: 4].
*   **Methodology Question:** Is the label window (`is_declining_label`) perfectly separated from the feature aggregation windows (like `impressions_90d` or `clicks_90d`)[cite: 4, 5]? To avoid leakage caused by overlapping windows, we must draw the timeline and ensure that, if the label is established over the last 30 days, only `*_prev30` columns and earlier are considered legal features[cite: 2, 4].

**Finding 2:** Missingness in the initial dataset varies drastically based on `content_type`, with some categories having nearly 100% missing keyword data, and `avg_position = 0` means "no data", not rank zero[cite: 4].
*   **Methodology Question:** Wouldn't a blind `fillna(0)` inject a hidden category signal directly into the model[cite: 4]? The validation design will fail to catch this if explicit `has-flags` are not added, to ensure the model isn't just learning an old system rule instead of a real-world pattern[cite: 2, 4].

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


**Validation Design Audit:** We use the grouped split by client (`GroupShuffleSplit`)[cite: 5] because the current dataset (`content_refresh_anonymized.csv`) consolidates metrics into a static 90-day window[cite: 4]. To perfectly simulate the production environment via a strict Time-Aware Split (train on the past, test on the future), the model would need to migrate to the `fact_content_daily_performance_sample` dataset from the warehouse, using the `report_date` column to define a chronological cutoff point alongside client isolation[cite: 2, 4].

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

# Load the starter FlyRank dataset[cite: 4, 5]
df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')

# The label is derived from trend_direction[cite: 4, 5]
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
base_rate = df['is_declining_label'].mean()

# Has-Flags creation for explicit mapping of missing data
# avg_position = 0 means "no data", so we create a flag to identify this absence[cite: 4]
df['has_search_data'] = (df['avg_position'] > 0).astype(int)
# The dataset has missing data linked to content_type (e.g., missing word_count)[cite: 4]
df['has_word_count'] = df['word_count'].notna().astype(int)

# Strictly observable Week 5 features, plus the new flags[cite: 5]
feature_cols = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'avg_position', 'ctr', 'content_age_days', 'days_since_last_update',
    'word_count', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'has_search_data', 'has_word_count'
]

X = df[feature_cols].fillna(0) # The fillna(0) is now safe thanks to the has-flags[cite: 4]
y = df['is_declining_label']

# 1. DISHONEST SPLIT (Random)
# A random split allows the model to memorize client-specific characteristics and fake skill[cite: 2].
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X, y, test_size=0.2, random_state=42)
model_rand = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_train_rand, y_train_rand)
pr_auc_rand = average_precision_score(y_test_rand, model_rand.predict_proba(X_test_rand)[:, 1])

# 2. HONEST SPLIT (Grouped by Client ID)
# Keeps domain dynamics isolated to test if the model generalizes to unseen clients[cite: 2, 4, 5].
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))

model_group = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X.iloc[train_idx], y.iloc[train_idx])
pr_auc_group = average_precision_score(y.iloc[test_idx], model_group.predict_proba(X.iloc[test_idx])[:, 1])

print(f"Base Rate (Label): {base_rate:.3f} (Always reported next to metrics)[cite: 2]")
print(f"PR-AUC (Random / Dishonest Split): {pr_auc_rand:.3f}")
print(f"PR-AUC (Grouped / Honest Split):    {pr_auc_group:.3f}")
print(f"Memorization Gap: {(pr_auc_rand - pr_auc_group):.3f}[cite: 2]")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Timeline and Window Overlap:**
The `is_declining_label` is derived from the `trend_direction` column[cite: 4, 5]. If this trend is calculated by comparing the last 30 days with the previous 30 days, using 90-day aggregated features (like `impressions_90d`) creates an overlap where the feature already "knows" part of the outcome[cite: 2]. The definitive architectural fix requires aligning features to strictly reflect the period before the label measurement window[cite: 2, 4].

In [ ]:
# The label trap: trend_direction and trend_pct can NEVER be features, as the label is derived from them[cite: 4, 5].
# We will deliberately ADD a leaky feature and watch the metric jump close to 1.0[cite: 2].

# Create intentionally leaky data
X_leaky = df[feature_cols + ['trend_pct']].fillna(0)

# Train with the honest split, but using the corrupted data
model_leaky = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
pr_auc_leaky = average_precision_score(y.iloc[test_idx], model_leaky.predict_proba(X_leaky.iloc[test_idx])[:, 1])

print(f"Clean Model PR-AUC:  {pr_auc_group:.3f}[cite: 5]")
print(f"Leaky Model PR-AUC (with trend_pct): {pr_auc_leaky:.3f}[cite: 2, 4]")

# --- Audit Checklist Completed ---
# [x] Timeline: 'impressions_90d' and decline label audited to avoid overlapping windows[cite: 2, 4].
# [x] Label sibling columns: 'trend_direction' and 'trend_pct' removed from features[cite: 2, 4].
# [x] Entity split: 'client_id' validated, ensuring zero intersection between train and test sets[cite: 2, 4, 5].

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

*   **Original Claim (Bold):** Our Random Forest model perfectly identified the traffic decline of any page, completely outperforming the Week 4 baseline rules, and must replace the current ranking system immediately.
*   **Rewritten Claim (Safe Language):** We measured that the Random Forest model offers a valid directional signal to identify declining content under a grouped validation; it is designed to output calibrated probabilities as decision-support for ranking the human review queue[cite: 5].

**Note on Feature Importance and Rate Limits:**
It is critical to note that metrics like `scroll_rate` and `ai_traffic_pct` can legitimately exceed 100 in the FlyRank system, since the numerator and denominator come from different measurement systems[cite: 4]. The model interprets this data as valid signals of engagement intensity, not errors. Furthermore, rate columns are percentages multiplied by 100 (e.g., `ctr = 0.76` means `0.76%`)[cite: 4], ensuring the algorithm's weights are not distorted by incorrect scales.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.